In [141]:
from cmath import sqrt

import torch

torch.__version__

'2.14.0+cu130'

In [142]:
import os

os.getcwd()

'C:\\LANG_CHAIN_2026\\2026-05-19_KDT_lang_chain\\02_self_study\\computer_vision'

In [144]:
from torchvision import transforms

transform_dict = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(32, (0.8, 1.0)),
        transforms.RandomAffine(10, (0.1, 0.1), (1.0, 1.0), (5, 5)),
        transforms.RandomVerticalFlip(p=0.5),

        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616)
        )
    ]),
    "val": transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616)
        )
    ])
}

In [145]:
from torchvision.datasets import CIFAR10

train_dataset = CIFAR10(
    root="./data",
    train=True,
    download=False,
    # transform=transforms.ToTensor()
    transform=transform_dict["train"]
)

test_dataset = CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform_dict["val"]
)

In [146]:
from torch.utils.data import DataLoader

data_loader = {
    "train": DataLoader(
        dataset=train_dataset,
        batch_size=128,
        shuffle=True,
        num_workers=4,
        persistent_workers=True,
        pin_memory=True,
    ),
    "val": DataLoader(
        dataset=test_dataset,
        batch_size=128,
        shuffle=False,
        num_workers=4,
        persistent_workers=True,
        pin_memory=True
    )
}

In [147]:
# images, _ = next(iter(data_loader["train"]))
# images.shape, images.mean(dim=(0, 2, 3)), images.dtype, type(images.shape[0])

In [148]:
# images, _ = next(iter(data_loader["train"]))
# (images.mean(dim=(2, 3)) / total_mean).detach().cpu()

In [149]:
# import math
#
# cumulative_mean = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32)
# total_count = 0
#
# for idx, (images, _) in enumerate(data_loader["train"]):
#     # print(f"batch num: {idx+1}")
#     cumulative_mean += images.mean(dim=(0, 2, 3)) * images.shape[0]
#     total_count += images.shape[0]
#
# total_mean = cumulative_mean / total_count
# print(f"total_mean: {total_mean}")
#
# var = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32)
# total_pixel_count = 0
#
# for idx, (images, _) in enumerate(data_loader["train"]):
#     diff = images - total_mean.view(1, 3, 1, 1)
#
#     var += (diff ** 2).sum(dim=(0, 2, 3))
#
#     B, C, H, W = images.shape
#
#     total_pixel_count += B * W * H
#
# var = var /  total_pixel_count
# print(f"total_std: {torch.sqrt(var)}")

In [150]:
def run_epoch(model, data_loader, criterion, optimizer=None):
    device = next(model.parameters()).device

    is_training = optimizer != None

    context = torch.enable_grad() if is_training else torch.inference_mode()

    if is_training:
        model.train()
    else:
        model.eval()

    total_count = 0
    total_loss = 0
    total_acc = 0

    with context:
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            if is_training:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_count += batch_size
            total_acc += (logits.argmax(dim=1) == labels).sum().item()
            total_loss += loss.item() * batch_size

    return total_loss / total_count, total_acc / total_count


In [151]:
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # b, 64, 16, 16
            nn.AvgPool2d(2),

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # b, 128, 8, 8
            nn.AvgPool2d(2),

            nn.Conv2d(128, 128, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # b, 128, 2, 2
            nn.AdaptiveAvgPool2d(2),
        )

        self.features = nn.Sequential(
            nn.Flatten(),

            nn.Linear(512, 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.features(x)


In [152]:
import torch.nn as nn
import torch.optim as optim

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MyModel(10).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

In [153]:
from torch.utils.tensorboard import SummaryWriter

EPOCHS = 50

writer = SummaryWriter(
    log_dir="./runs/cnn_default_v1"
)

for epoch in range(1, EPOCHS+1):
    train_loss, train_acc = run_epoch(model, data_loader["train"], criterion, optimizer)

    val_loss, val_acc = run_epoch(model, data_loader["val"], criterion)

    # writer
    writer.add_scalars(
        "Loss",
        {
            "train": train_loss,
            "val": val_loss
        }
        ,
        epoch
    )
    writer.add_scalars(
        "Accuracy",
        {
            "train": train_acc,
            "val": val_acc
        },
        epoch
    )

    if epoch % 10 == 0:
        print(f"""
epoch: {epoch}
\t| train loss: {train_loss}\t| train acc: {train_acc}
\t| val loss: {val_loss}\t| val acc: {val_acc}
""")

writer.close()


epoch: 10
	| train loss: 0.8671698500442505	| train acc: 0.68844
	| val loss: 0.8516269151687622	| val acc: 0.704


epoch: 20
	| train loss: 0.7008528536987305	| train acc: 0.75114
	| val loss: 0.7452235062599182	| val acc: 0.7446


epoch: 30
	| train loss: 0.6234008372116089	| train acc: 0.77958
	| val loss: 0.818909949016571	| val acc: 0.7364


epoch: 40
	| train loss: 0.5644619422054291	| train acc: 0.8003
	| val loss: 0.6735819551944733	| val acc: 0.7748


epoch: 50
	| train loss: 0.5217870490837098	| train acc: 0.8161
	| val loss: 0.7996945035934448	| val acc: 0.7492

